# B10_long. Generalization: elapsed times beyond training

Up to 150,000 yr, in one call and chained, from the start terrain.

See `docs/plans/experiments-plan.md`, entry B10_long. Results: `artifacts/results/erosion_long/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
(cd datagen && uv run erosion-datagen generate erosion_master/main_256)
(cd datagen && uv run erosion-datagen generate erosion_master/long_256)
```

Cluster:

```sh
sbatch slurm/datagen.sbatch erosion_master/main_256
sbatch slurm/datagen.sbatch erosion_master/long_256
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped.

Local:

```sh
uv run erosion-sim train erosion_final_seed0
```

Cluster:

```sh
sbatch slurm/train.sbatch erosion_final_seed0
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate erosion_long erosion_final_seed0
uv run erosion-sim evaluate erosion_long no_change
uv run erosion-sim evaluate erosion_long analytical
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch erosion_long erosion_final_seed0
sbatch slurm/evaluate.sbatch erosion_long no_change
sbatch slurm/evaluate.sbatch erosion_long analytical
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/erosion.md`](../../docs/guides/erosion.md).

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "erosion_long"  # configs/evaluations/erosion_long.yaml
MODELS = ["erosion_final_seed0"]  # trained models
BASELINES = ["no_change", "analytical"]  # configs/predictors/
PREDICTORS = MODELS + BASELINES

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Training

In [ ]:
for name in MODELS:
    print(name, model_dir(name))
    plot_history(json.loads((model_dir(name) / "history.json").read_text()))

## Every metric, over all test pairs

In [ ]:
plots.table(results)

## One call against chained calls

In [ ]:
plots.table(results)  # rows "<model> chained <n>x<years>" are scored on the pairs at n × years

## Breakdowns

Each factor with more than one group among the test pairs.

In [ ]:
for factor in results.scalars["by"]:
    plots.by_table(results, factor, "skill")
    print()

In [ ]:
for factor in results.scalars["by"]:
    plots.plot_by(results, factor, "skill")

## Detail and examples

In [ ]:
plots.plot_spectrum(results)
plots.plot_examples(results)

## Conclusion

_Written after reading the results above._